# CRISP-DM Fase 2: Data Understanding - Exploración Informática y Estadística (EDA)
## Dataset: SIPSA Precios de Insumos y Fertilizantes Agrícolas (DANE)

**Ecosistema**: AgroStats Intelligence Platform (`AgroStatsApp`)  
**Metodología**: CRISP-DM (Fase 2: Exploratory Data Analysis) | **Fase PDCO**: DEVELOPMENT  
**Estándares Aplicados**: ISO/IEC 25010 (Data Quality Metrics), Control Estadístico de Procesos (SPC Nelson Rules).

---

### 1. Objetivos del Análisis Exploratorio de Datos (EDA)
El EDA sistemático persigue dos metas complementarias:
1. **Diagnóstico Informático**: Identificar tipos de datos, consumo de memoria, cardinalidad y patrones de datos faltantes bajo la taxonomía de Donald Rubin:
   * **MCAR** (*Missing Completely at Random*): La ausencia es independiente de cualquier variable observada o no observada.
   * **MAR** (*Missing at Random*): La probabilidad de ausencia depende de otras variables observables.
   * **MNAR** (*Missing Not at Random*): La ausencia depende del valor intrínseco de la propia variable no observada.
2. **Diagnóstico Estadístico**: Evaluar la forma funcional de la distribución empírica mediante los **cuatro momentos estadísticos de Pearson** y detectar valores atípicos mediante **Control Estadístico de Procesos (SPC)**.

---


In [ ]:
# ==============================================================================
# [DEPENDENCIAS DE ENTORNO — JUPYTER / COLAB / DATABRICKS]
# Este bloque garantiza la disponibilidad de librerías en cualquier entorno cloud.
# ==============================================================================
%pip install -q pandas numpy requests python-dotenv openpyxl pypdf pyarrow pyreadstat matplotlib seaborn scipy statsmodels scikit-learn duckdb pydantic


In [ ]:
# ==============================================================================
# [CONFIGURACIÓN DEL KERNEL Y RESOLUCIÓN DE RUTAS DEL PROYECTO (PEP 8)]
# Importación estándar y resolución dinámica del path para src/
# ==============================================================================
import os
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

# Resolver raíz del proyecto para importar módulos de src/
CURRENT_DIR = Path(".").resolve()
APP_ROOT = CURRENT_DIR.parent.parent
if str(APP_ROOT) not in sys.path:
    sys.path.insert(0, str(APP_ROOT))

import pandas as pd
import numpy as np

print(f"✅ Entorno inicializado exitosamente. Raíz del proyecto: {APP_ROOT}")


### 2. Perfilamiento Informático y Completitud
Examinamos la estructura tabular, el footprint en memoria RAM y el porcentaje de completitud por columna para planificar las reglas de limpieza y validación.


In [ ]:
# ==============================================================================
# [FASE 3: PERFILAMIENTO INFORMÁTICO Y CONSUMO DE MEMORIA]
# ==============================================================================
from src.database.db_manager import DatabaseManager

parquet_file = APP_ROOT / "data/processed/sipsa_insumos.parquet"
if parquet_file.exists():
    df = pd.read_parquet(parquet_file)
else:
    db = DatabaseManager(str(APP_ROOT / "data" / "processed" / "agrostats_lakehouse.db"))
    df = db.query("SELECT * FROM sipsa_insumos")

print(f"📊 Dimensiones del Dataset: {df.shape[0]:,} filas x {df.shape[1]} columnas")
print(f"💾 Consumo en Memoria RAM: {df.memory_usage(deep=True).sum() / (1024 * 1024):.2f} MB")

print("\n--- DIAGNÓSTICO DE COMPLETITUD Y NULOS POR VARIABLE ---")
missing_df = pd.DataFrame({
    "tipo_dato": df.dtypes,
    "nulos": df.isnull().sum(),
    "nulos_pct": (df.isnull().sum() / len(df)) * 100,
    "valores_unicos": df.nunique()
})
display(missing_df)


### 3. Estimación de Momentos Estadísticos y Regla 1 de Nelson
Para la variable cuantitativa principal, calculamos:
1. **Media Aritmética ($\\mu$)** y **Desviación Estándar ($\\sigma$)**:
   $$\\mu = \\frac{1}{N} \\sum_{i=1}^N X_i, \quad \\sigma = \\sqrt{\\frac{1}{N-1} \\sum_{i=1}^N (X_i - \\mu)^2}$$
2. **Asimetría ($S$) y Curtosis ($K$)**:
   $$S = \\frac{\\frac{1}{N} \\sum (X_i - \\mu)^3}{\\sigma^3}, \quad K = \\frac{\\frac{1}{N} \\sum (X_i - \\mu)^4}{\\sigma^4} - 3$$
3. **Test de Normalidad Jarque-Bera**:
   $$JB = \\frac{N}{6} \\left( S^2 + \\frac{K^2}{4} \\right)$$
4. **Control Estadístico de Procesos (SPC - Regla 1 de Nelson)**: Identifica observaciones a más de tres desviaciones estándar de la media ($|z| > 3$).


In [ ]:
# ==============================================================================
# [FASE 3: DIAGNÓSTICO ESTADÍSTICO DE DISTRIBUCIONES Y CONTROL SPC]
# ==============================================================================
from scipy import stats
from src.modeling.sarimax_model import AgroModeler

target_col = "indice_total"
if target_col in df.columns and pd.api.types.is_numeric_dtype(df[target_col]):
    s = df[target_col].dropna()
    
    mean_val = float(s.mean())
    std_val = float(s.std())
    skew_val = float(stats.skew(s))
    kurt_val = float(stats.kurtosis(s))
    jb_stat, jb_pval = stats.jarque_bera(s)
    
    print("--- 1. MOMENTOS ESTADÍSTICOS DE PEARSON ---")
    print(f"• Media Aritmética (Media): {mean_val:,.4f}")
    print(f"• Desviación Estándar (Sigma): {std_val:,.4f}")
    print(f"• Coeficiente de Asimetría (Skewness): {skew_val:.4f} ({'Asimetría Positiva' if skew_val > 0 else 'Asimetría Negativa'})")
    print(f"• Curtosis: {kurt_val:.4f} ({'Leptocúrtica (Colas Pesadas)' if kurt_val > 0 else 'Platicúrtica'})")
    print(f"• Test Jarque-Bera: p-value = {jb_pval:.4e} ({'Rechaza normalidad estricta' if jb_pval < 0.05 else 'Compatible con distribución normal'})")
    
    print("\n--- 2. DETECCIÓN DE ANOMALÍAS (SPC REGLA 1 DE NELSON) ---")
    anomalies = AgroModeler.detect_nelson_anomalies(s)
    print(f"• Observaciones fuera de control (|z| > 3): {anomalies.sum()} de {len(s)} ({(anomalies.sum()/len(s)):.2%})")
else:
    print("ℹ️ La columna 'indice_total' es textual o estructurada. Evaluando cardinalidad categórica:")
    print(df.describe(include='all'))


### 4. Visualización Diagnóstica de Distribución y Vallas de Tukey
Construimos un panel dual con un histograma con ajuste de densidad kernel (KDE) y un diagrama de caja (Boxplot) señalando los cuartiles $Q_1$, $Q_2$ (mediana) y $Q_3$, junto con los límites de Tukey ($1.5 \\times IQR$).


In [ ]:
# ==============================================================================
# [FASE 3: VISUALIZACIONES DIAGNÓSTICAS EDA]
# ==============================================================================
import matplotlib.pyplot as plt
import seaborn as sns

target_col = "indice_total"
if target_col in df.columns and pd.api.types.is_numeric_dtype(df[target_col]):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    s = df[target_col].dropna()
    
    # Subplot 1: Histograma y KDE
    sns.histplot(s, kde=True, ax=axes[0], color="#2b5c8f", bins=30)
    axes[0].axvline(s.mean(), color="red", linestyle="--", label=f"Media: {s.mean():,.2f}")
    axes[0].axvline(s.median(), color="green", linestyle="-", label=f"Mediana: {s.median():,.2f}")
    axes[0].set_title("Distribución empírica de indice_total")
    axes[0].legend()
    
    # Subplot 2: Boxplot con límites IQR
    sns.boxplot(x=s, ax=axes[1], color="#52b788")
    axes[1].set_title("Boxplot y Detección de Outliers (Vallas de Tukey)")
    
    plt.tight_layout()
    plt.show()


### 5. Diagnóstico Final del EDA y Recomendaciones para Data Preparation
1. **Tratamiento de Asimetrías**: Si el test Jarque-Bera rechaza la normalidad ($p < 0.05$), se hace imperativo el uso de estimadores no paramétricos (Mediana, Theil-Sen, Spearman).
2. **Siguiente Paso**: Proceder a `04_ingestion_como_dataframe.ipynb` para aplicar los Quality Gates automáticos de tipificación y completitud.
